In [33]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import sparse
from sklearn.cluster import KMeans
from sklearn.manifold import TSNE
from sklearn.metrics import (
    normalized_mutual_info_score, 
    adjusted_rand_score,
    silhouette_score
)
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import normalize
import re
from collections import defaultdict, Counter

In [34]:
plt.rcParams['font.sans-serif'] = ['DejaVu Sans', 'Arial']
plt.rcParams['axes.unicode_minus'] = False
sns.set_style("whitegrid")

In [35]:
class ASHD:
    def __init__(self, n_clusters=7, embedding_dim=256, n_iter=25, 
                 alpha=0.5, lambda_reg=0.01, gamma=0.1):
        self.n_clusters = n_clusters
        self.embedding_dim = embedding_dim
        self.n_iter = n_iter
        self.alpha = alpha              
        self.lambda_reg = lambda_reg    # L2 
        self.gamma = gamma              # Max-cut
        self.eps = 1e-10
        self.weight_history = []

    def load_data(self, papers_file, citations_file, topics_file): #papers_dataset.txt, citations.txt, topics.txt
        self.paper_features = {}
        paper_list = []
        
        with open(papers_file, 'r') as f:
            for line in f:
                if ';paper;' in line: # find paper 
                    parts = line.strip().split(';') # get id
                    pid = parts[0]
                    paper_list.append(pid)
                    
                    if len(parts) > 2:
                        feature_dict = {} # save each paper
                        for item in parts[2].split(','):
                            if ':' in item:
                                word_id, weight = item.split(':')
                                feature_dict[word_id] = float(weight)
                        self.paper_features[pid] = feature_dict
        
        self.paper_list = sorted(list(set(paper_list)))
        self.paper_to_idx = {pid: i for i, pid in enumerate(self.paper_list)}
        n_nodes = len(self.paper_list)
        print(f"Papers: {n_nodes}")

        # true label
        true_labels = self._extract_ground_truth(topics_file) # the real topic

        # positive edge: one paper and all papers it cites
        pos_edges = self._build_positive_hyperedges(citations_file)
        print(f"Positive hyperedges (citations): {len(pos_edges)}")
        
        # negative edges: same topic, but non-citation
        neg_edges = self._build_negative_hyperedges_v3(citations_file, topics_file)
        print(f"Negative hyperedges (peers): {len(neg_edges)}")
        return n_nodes, pos_edges, neg_edges, true_labels

    def _extract_ground_truth(self, topics_file):
        paper_topics = {}
        topic_to_id = {}
        self.topic_names = {}
        current_topic_id = 0
        
        with open(topics_file, 'r') as f:
            content = f.read()
            topics = re.findall(r'(\w+)\((\d+)\)=1', content)
            
            for topic, pid in topics:
                if pid in self.paper_to_idx:
                    if topic not in topic_to_id:
                        topic_to_id[topic] = current_topic_id
                        self.topic_names[current_topic_id] = topic
                        current_topic_id += 1
                    
                    if pid not in paper_topics:
                        paper_topics[pid] = topic_to_id[topic]
        
        true_labels = np.full(len(self.paper_list), -1, dtype=int)
        for i, pid in enumerate(self.paper_list):
            if pid in paper_topics:
                true_labels[i] = paper_topics[pid]
        
        return true_labels

    def _build_positive_hyperedges(self, citations_file):
        pos_edges = defaultdict(set)
        with open(citations_file, 'r') as f:
            content = f.read()
            citations = re.findall(r'Cite\((\d+),(\d+)\)=1', content)
            for source, target in citations:
                if source in self.paper_to_idx and target in self.paper_to_idx:
                    pos_edges[source].add(self.paper_to_idx[source])
                    pos_edges[source].add(self.paper_to_idx[target])
        
        return [list(nodes) for nodes in pos_edges.values() if len(nodes) > 1]

    def _build_negative_hyperedges_v3(self, citations_file, topics_file):
        # build citation graph
        citation_graph = defaultdict(set)
        with open(citations_file, 'r') as f:
            content = f.read()
            citations = re.findall(r'Cite\((\d+),(\d+)\)=1', content)
            for source, target in citations:
                if source in self.paper_to_idx and target in self.paper_to_idx:
                    citation_graph[source].add(target)
                    citation_graph[target].add(source)
        
        # group by topics
        topic_groups = defaultdict(set)
        with open(topics_file, 'r') as f:
            content = f.read()
            topics = re.findall(r'(\w+)\((\d+)\)=1', content)
            for topic, pid in topics:
                if pid in self.paper_to_idx:
                    topic_groups[topic].add(pid)
        
        # non-citation pairs
        neg_edges = []
        for topic, papers in topic_groups.items():
            papers_list = list(papers)
            if len(papers_list) < 2:
                continue
            
            # Most 150 negative edges in each topic
            sample_size = min(150, len(papers_list))
            sampled = np.random.choice(papers_list, sample_size, replace=False)
            
            count = 0
            max_pairs_per_topic = 100
            
            for i in range(len(sampled)):
                if count >= max_pairs_per_topic:
                    break
                for j in range(i+1, len(sampled)):
                    p1, p2 = sampled[i], sampled[j]
                    # never cites
                    if p2 not in citation_graph[p1] and p1 not in citation_graph[p2]:
                        neg_edges.append([self.paper_to_idx[p1], self.paper_to_idx[p2]])
                        count += 1
                        if count >= max_pairs_per_topic:
                            break
        
        return neg_edges

    def _features_to_embedding(self):
        # words to vectors
        all_words = set()
        for features in self.paper_features.values():
            all_words.update(features.keys())
        
        word_to_idx = {word: i for i, word in enumerate(sorted(all_words))}
        n_words = len(word_to_idx)
        
        # sparse feature matrix
        rows, cols, data = [], [], []
        for i, pid in enumerate(self.paper_list):
            if pid in self.paper_features:
                for word, weight in self.paper_features[pid].items():
                    if word in word_to_idx:
                        rows.append(i)
                        cols.append(word_to_idx[word])
                        data.append(weight)
        
        feature_matrix = sparse.csr_matrix(
            (data, (rows, cols)), 
            shape=(len(self.paper_list), n_words)
        )
        
        from sklearn.random_projection import GaussianRandomProjection
        projector = GaussianRandomProjection(n_components=self.embedding_dim, random_state=42)
        X_init = projector.fit_transform(feature_matrix.toarray())
        
        # norm
        X_init = normalize(X_init, norm='l2', axis=1)
        return X_init

    def build_incidence_matrix(self, n_nodes, pos_edges, neg_edges):
        all_edges = pos_edges + neg_edges
        n_edges = len(all_edges)
        
        rows, cols = [], []
        signs = []
        
        for j, edge_nodes in enumerate(all_edges):
            is_pos = j < len(pos_edges)
            signs.append(1.0 if is_pos else -1.0)
            for node_idx in edge_nodes:
                rows.append(node_idx)
                cols.append(j)
        
        H = sparse.csr_matrix(
            (np.ones(len(rows)), (rows, cols)), 
            shape=(n_nodes, n_edges)
        )
        
        self.n_pos_edges = len(pos_edges)
        self.n_neg_edges = len(neg_edges)
        
        print(f"Incidence matrix: {n_nodes} nodes × {n_edges} hyperedges")
        
        return H, np.array(signs)

    def fit_predict(self, H, initial_signs):
        n_nodes, n_edges = H.shape
        if self.paper_features:
            X = self._features_to_embedding()
        else:
            X = np.random.normal(0, 0.01, (n_nodes, self.embedding_dim))
            X = normalize(X, norm='l2', axis=1)
        
        W = initial_signs.astype(float).copy()
        
        d_e = np.array(H.sum(axis=0)).flatten() + self.eps
        De_inv = sparse.diags(1.0 / d_e)
        self.weight_history = []
        
        for t in range(self.n_iter):
            W_abs = np.abs(W)
            d_v = np.array(H.dot(W_abs)).flatten() + self.eps
            Dv_inv_sqrt = sparse.diags(1.0 / np.sqrt(d_v))
            
            # diffusion
            W_diag = sparse.diags(W)
            temp = Dv_inv_sqrt.dot(X)
            temp = H.T.dot(temp)
            temp = De_inv.dot(temp)
            temp = W_diag.dot(temp)
            temp = H.dot(temp)
            X_new = Dv_inv_sqrt.dot(temp)
            
            # L2 norm
            X = X_new - self.lambda_reg * X_new
            X = normalize(X, norm='l2', axis=1)
            
            # weight record
            self.weight_history.append(W.copy())
            
            # update weight
            if t > 0 and t % 2 == 0:
                for j in range(n_edges):
                    idx = H.getcol(j).indices
                    if len(idx) > 1:
                        sims = cosine_similarity(X[idx])
                        avg_sim = (np.sum(sims) - len(idx)) / (len(idx) * (len(idx) - 1) + self.eps)
                        
                        if W[j] > 0:
                            W[j] = np.clip(W[j] + self.alpha * avg_sim, 0.1, 1.0)
                        else:  
                            W[j] = np.clip(W[j] - self.alpha * avg_sim, -1.0, -0.1)

        # clustering
        labels = KMeans(
            n_clusters=self.n_clusters, 
            n_init=50, 
            max_iter=500,
            random_state=42
        ).fit_predict(X)
        
        return labels, X
    

    # picture
    def plot_tsne(self, embeddings, pred_labels, true_labels, save_path='tsne.png'):
        tsne = TSNE(n_components=2, random_state=42, perplexity=30)
        embeddings_2d = tsne.fit_transform(embeddings)
        
        fig, axes = plt.subplots(1, 2, figsize=(16, 6))
        
        scatter1 = axes[0].scatter(embeddings_2d[:, 0], embeddings_2d[:, 1], c=pred_labels, cmap='tab10', s=20, alpha=0.7)
        axes[0].set_title('ASHD Optimized Clustering (Target: ARI/NMI ≥ 0.8)', fontsize=14, fontweight='bold')
        axes[0].set_xlabel('t-SNE Dimension 1')
        axes[0].set_ylabel('t-SNE Dimension 2')
        plt.colorbar(scatter1, ax=axes[0], label='Cluster ID')
        
        valid_mask = true_labels != -1
        scatter2 = axes[1].scatter(embeddings_2d[valid_mask, 0], embeddings_2d[valid_mask, 1], c=true_labels[valid_mask], cmap='tab10', s=20, alpha=0.7)
        axes[1].set_title('Ground Truth Topics', fontsize=14, fontweight='bold')
        axes[1].set_xlabel('t-SNE Dimension 1')
        axes[1].set_ylabel('t-SNE Dimension 2')
        plt.colorbar(scatter2, ax=axes[1], label='Topic ID')
        
        plt.tight_layout()
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        plt.close()

    def plot_cluster_distribution(self, pred_labels, true_labels, save_path='distribution.png'):
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))

        unique_pred, counts_pred = np.unique(pred_labels, return_counts=True)
        axes[0].bar(unique_pred, counts_pred, color='steelblue', alpha=0.7, edgecolor='black')
        axes[0].set_title('ASHD Optimized Cluster Distribution', fontsize=12, fontweight='bold')
        axes[0].set_xlabel('Cluster ID')
        axes[0].set_ylabel('Number of Papers')
        axes[0].grid(axis='y', alpha=0.3)
        
        valid_true = true_labels[true_labels != -1]
        unique_true, counts_true = np.unique(valid_true, return_counts=True)
        topic_labels = [self.topic_names.get(tid, f'T{tid}') for tid in unique_true]
        axes[1].bar(range(len(unique_true)), counts_true, color='coral', alpha=0.7, edgecolor='black')
        axes[1].set_xticks(range(len(unique_true)))
        axes[1].set_xticklabels(topic_labels, rotation=45, ha='right')
        axes[1].set_title('Ground Truth Topic Distribution', fontsize=12, fontweight='bold')
        axes[1].set_xlabel('Topic')
        axes[1].set_ylabel('Number of Papers')
        axes[1].grid(axis='y', alpha=0.3)
        
        plt.tight_layout()
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        plt.close()

In [ ]:
if __name__ == "__main__":
    ashd = ASHD(n_clusters=31,embedding_dim=256,n_iter=25,alpha=0.5,lambda_reg=0.01,gamma=0.1)             
    
    # load data
    n_nodes, pos_edges, neg_edges, true_labels = ashd.load_data(
        'papers_dataset.txt', 
        'citations.txt', 
        'topics.txt'
    )
    # build signed hypergraph
    H, initial_signs = ashd.build_incidence_matrix(n_nodes, pos_edges, neg_edges)
    # ASHD
    pred_labels, embeddings = ashd.fit_predict(H, initial_signs)
    
    valid_mask = true_labels != -1
    ari = adjusted_rand_score(true_labels[valid_mask], pred_labels[valid_mask])
    nmi = normalized_mutual_info_score(true_labels[valid_mask], pred_labels[valid_mask])
    print(f"  ARI:        {ari:.4f}")
    print(f"  NMI:        {nmi:.4f}")
    
    ashd.plot_tsne(embeddings, pred_labels, true_labels)
    ashd.plot_cluster_distribution(pred_labels, true_labels)

Papers: 11881
Positive hyperedges (citations): 9548
Negative hyperedges (peers): 7845
Incidence matrix: 11881 nodes × 17393 hyperedges
  ARI:        0.1309
  NMI:        0.3427
